# EX1:
The n-dimensional tensor mastery challenge: Combine the `Head` and `MultiHeadAttention` into one class that processes all the heads in parallel, treating the heads as another batch dimension (answer is in nanoGPT).

In [4]:
import torch
import torch.nn as nn
from torch.nn import functional as F

In [ ]:
batch_size = 4
block_size = 8
n_embd = 32

In [64]:
torch.manual_seed(1337)

class Head(nn.Module):
    """one head of self-attention (kept as the single-head reference)"""

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)  # (B, T, head_size)
        q = self.query(x)  # (B, T, head_size)

        # compute attention scores - affinities
        wei = (
            q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5
        )  # (B, T, hs) @ (B, hs, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))  # (B, T, T)
        wei = F.softmax(wei, dim=-1)  # (B, T, T)

        # perform the weighted aggregation of the values
        v = self.value(x)  # (B, T, head_size)
        out = wei @ v  # (B, T, T) @ (B, T, head_size) -> (B, T, head_size)
        return out

class MultiHeadAttention(nn.Module):
    """multiple heads of self-attention in parallel"""

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.proj(out)
        return out


class MultiHeadAttentionFused(nn.Module):
    """all heads of self-attention computed in parallel (Head + MHA merged, nanoGPT form).

    Key idea: instead of looping over `num_heads` separate Head modules, do ONE big
    projection of width n_embd (= num_heads * head_size) for each of k/q/v, then split
    the channel axis into (num_heads, head_size) and slide num_heads next to the batch
    dim. The heads then just ride along as an extra batch dimension: the matmuls
    broadcast over (B, num_heads) and only act on the trailing (T, head_size) dims.
    """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.num_heads = num_heads
        self.head_size = head_size

        # one fused projection per role; output width = num_heads * head_size (== n_embd here)
        self.key = nn.Linear(n_embd, num_heads * head_size, bias=False)
        self.query = nn.Linear(n_embd, num_heads * head_size, bias=False)
        self.value = nn.Linear(n_embd, num_heads * head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.proj = nn.Linear(num_heads * head_size, n_embd)

    def forward(self, x):
        B, T, C = x.shape
        nh, hs = self.num_heads, self.head_size

        # project, then carve the channel dim into (nh, hs) and move heads beside the batch dim.
        # (B, T, nh*hs) -view-> (B, T, nh, hs) -transpose(1,2)-> (B, nh, T, hs)
        k = self.key(x).view(B, T, nh, hs).transpose(1, 2)  # (B, nh, T, hs)
        q = self.query(x).view(B, T, nh, hs).transpose(1, 2)  # (B, nh, T, hs)
        v = self.value(x).view(B, T, nh, hs).transpose(1, 2)  # (B, nh, T, hs)

        # attention: matmul/softmax act on the last two dims; (B, nh) ride along as batch dims
        wei = (
            q @ k.transpose(-2, -1) * hs**-0.5
        )  # (B, nh, T, hs) @ (B, nh, hs, T) -> (B, nh, T, T)
        wei = wei.masked_fill(
            self.tril[:T, :T] == 0, float("-inf")
        )  # (T, T) broadcasts over (B, nh)
        wei = F.softmax(wei, dim=-1)  # (B, nh, T, T)

        out = wei @ v  # (B, nh, T, T) @ (B, nh, T, hs) -> (B, nh, T, hs)

        # re-assemble heads side by side, reproducing the old cat(dim=-1) layout:
        # (B, nh, T, hs) -transpose(1,2)-> (B, T, nh, hs) -view-> (B, T, nh*hs)
        out = out.transpose(1, 2).contiguous().view(B, T, nh * hs)
        out = self.proj(out)  # (B, T, n_embd)
        return out


In [65]:
num_heads, head_size = 4, n_embd // 4
old = MultiHeadAttention(num_heads, head_size)
new = MultiHeadAttentionFused(num_heads, head_size)

# Copy NEW's fused weights into OLD's per-head slices.
# view(B,T,nh,hs) splits output channels as [head0 | head1 | ...], so rows
# [i*hs:(i+1)*hs] of the fused weight are head i's projection.
hs = head_size
with torch.no_grad():
    for i, h in enumerate(old.heads):
        sl = slice(i * hs, (i + 1) * hs)
        h.key.weight.copy_(new.key.weight[sl])
        h.query.weight.copy_(new.query.weight[sl])
        h.value.weight.copy_(new.value.weight[sl])
    old.proj.weight.copy_(new.proj.weight)
    old.proj.bias.copy_(new.proj.bias)

x = torch.randn(batch_size, block_size, n_embd)
y_new, y_old = new(x), old(x)

print("shapes:", y_new.shape, y_old.shape)
print("max abs diff:", (y_new - y_old).abs().max().item())
print("allclose:", torch.allclose(y_new, y_old, atol=1e-6))


shapes: torch.Size([4, 8, 32]) torch.Size([4, 8, 32])
max abs diff: 1.7881393432617188e-07
allclose: True
